# B3 — Fast Weight Meta-Learning (FWML)

Ana ağırlıklar donar, her bloğun çıktısına LoRA-rank fast weight (A_t = U·V) eklenir. Hebbian outer-product ile gradient-free güncelleme.

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/B3_FWML_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "B3_FWML",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    # FWML: ana ağırlıklar donuk; bu LR sadece scheduler tutarlılığı için
    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  0,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (FWML)
    "fw_rank":          32,
    "fw_hebbian_lr":    0.01,
    "fw_decay":         0.99,
    "fw_freeze_slow":   True,
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Fast Weight Meta-Learning ───────────────────────
class FastWeightMetaLearning:
    requires_meta_loop = False
    owns_optimizer = False

    def __init__(self, model, params):
        self.model = model
        mp = params
        self.rank = int(mp.get("fw_rank", 32))
        self.hebbian_lr = float(mp.get("fw_hebbian_lr", 0.01))
        self.decay = float(mp.get("fw_decay", 0.99))
        self.freeze_slow = bool(mp.get("fw_freeze_slow", True))

        hidden = model.config.n_embd
        n_layers = model.config.n_layer
        device = next(model.parameters()).device
        self.fast_U = [nn.Parameter(torch.zeros(hidden, self.rank, device=device), requires_grad=False) for _ in range(n_layers)]
        self.fast_V = [nn.Parameter(torch.zeros(self.rank, hidden, device=device), requires_grad=False) for _ in range(n_layers)]

        if self.freeze_slow:
            for p in self.model.parameters():
                p.requires_grad = False
            # Embeddings + LM head'i de dondurduk; mini bir trainable head bırakalım:
            for p in self.model.lm_head.parameters():
                p.requires_grad = True

        self._last_inputs = {}
        self._handles = []
        for idx, block in enumerate(self.model.transformer.h):
            self._handles.append(block.register_forward_hook(self._make_hook(idx)))

    def set_mask_token_id(self, tid):
        pass

    def _make_hook(self, idx):
        def hook(_mod, inputs, output):
            x = inputs[0]
            self._last_inputs[idx] = x.detach()
            U = self.fast_U[idx]; V = self.fast_V[idx]
            fast_out = (x @ V.T) @ U.T
            if isinstance(output, tuple):
                return (output[0] + fast_out,) + output[1:]
            return output + fast_out
        return hook

    def adapt(self, context):
        was = self.model.training
        self.model.eval()
        with torch.no_grad():
            self.model(input_ids=context["input_ids"], attention_mask=context.get("attention_mask"))
            for idx, x in self._last_inputs.items():
                mean_h = x.mean(dim=(0, 1))
                key = mean_h[:self.rank]
                self.fast_U[idx].data += self.hebbian_lr * torch.outer(mean_h, key)
                self.fast_U[idx].data *= self.decay
                self.fast_V[idx].data *= self.decay
        if was: self.model.train()

    def meta_train_step(self, support, query):
        return 0.0

print("FastWeightMetaLearning tanımlandı.")

In [ ]:
def build_model_and_adapter(params):
    from transformers import GPT2LMHeadModel
    model = GPT2LMHeadModel.from_pretrained(params["model_name"])
    adapter = FastWeightMetaLearning(model, params)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"FWML → trainable {trainable:,} / total {total:,}")
    return model, adapter

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL: {result['final_metrics']['test_ppl']:.4f}")
print(f"Final test BPC: {result['final_metrics']['test_bpc']:.4f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")